# Behavioural analysis

All the analysis code lives in the `ppsanalysis` package. This notebook only
**drives** it: load the data, build the shared tables, run each hypothesis, read
the results.

```
ppsanalysis/
  config.py        paths, subject metadata, RT window, seeds, thresholds
  figures.py       publication-grade SVG settings and helpers
  io.py            Unity export loading and tidying
  qc.py            trial usability and data-quality checks
  stats_utils.py   sigmoid/AIC fits, bootstrap, permutation, classification
  pps.py           facilitation, near-far index, sigmoid boundary, Delta_PPS
  collision.py     PSE, Delta_coll, carryover, accuracy
  permutations.py  label-shuffling nulls for the single-case tests
  tables.py        the shared derived tables, built once
  hypotheses/
    h1  h1a  h2  h3  h4     Aim 1, young controls
    h5  h6   h6b  h7        Aim 2, the patient
    h8a h8b                 Aim 3, PPS and Hit-or-Miss together
```

Every hypothesis module has a `run(t)` function, a `report()` that prints the
numbers, and a `make_figure()` that draws them. Open any of them and read it
top to bottom: they are written as a sequence of numbered steps, and each step
says why it is there.

**Figures.** Every figure is saved as an editable SVG (and a PDF) in `figures/`.
They are drawn at real journal column width, so the font sizes are already correct
and you must not resize them afterwards.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import style
style.apply()

import ppsanalysis as pa
from ppsanalysis import config, figures
from ppsanalysis.hypotheses import h1, h1a, h2, h3, h4, h5, h6, h6b, h7, h8a, h8b

# Publication-grade defaults: vector SVG, editable text, bold titles,
# journal-sized fonts, no top/right frame lines.
figures.apply()
figures.FIGURE_DIR = "figures"

pd.set_option("display.max_columns", 120)

## 0.1 Configuration

Everything the analysis depends on is in `ppsanalysis/config.py`. Override it here.

Two settings deserve attention:

**`RT_MAX_MS` is now 1000, not 3000.** The old ceiling was longer than an entire
fast trial, so it let in presses made near the *end* of a trial as if they were
detections. Petrizzo et al. (2024) excluded RTs above 1000 ms on this exact task
and reported mean RTs of 250 to 320 ms.

**`DROP_POSITIONS` excludes distance levels entirely.** Dropping one is a
preregistration deviation. Leave it empty unless you have a stated reason.

In [ ]:
config.PILOT_DATA_DIR = "/Users/pamelavandenenden/Desktop/PPS/05_analysis/data/behavior"
config.OUT_DIR = "."

config.RT_MIN_MS = 100
config.RT_MAX_MS = 900
config.DROP_POSITIONS = ()          # e.g. (7,) to drop D7

config.YOUNG_CONTROL_SUBJECTS = ["theo", "franc"]
config.PATIENT_ID = None
config.MATCHED_CONTROL_ID = None

config.N_BOOT = 5000
config.RANDOM_SEED = 123

## 1. Load the data and build the shared tables

In [ ]:
pps_trials, collision_trials, subjects = pa.io.make_analysis_csvs(
    data_dir=config.PILOT_DATA_DIR,
    out_dir=config.OUT_DIR,
)

pps_trials = pa.qc.mark_pps_usable(
    pps_trials,
    rt_min_ms=config.RT_MIN_MS,
    rt_max_ms=config.RT_MAX_MS,
    drop_positions=config.DROP_POSITIONS,
)

# Build every derived table once, so no two hypotheses can silently disagree.
t = pa.tables.build(pps_trials, collision_trials, subjects)
pa.tables.describe(t)

## 1.1 Data quality

In [ ]:
pa.qc.quick_qc(t.pps_trials, t.collision_trials)

### Sanity check: is the distance axis the right way round?

`position_rank` must go UP as the stimulus gets FURTHER from the body. If it is
inverted, then `near` and `far` swap places and every facilitation sign flips,
silently.

The check: mean RT should **rise** with rank. A far-away stimulus helps you less,
so you are slower. If RT falls with rank, the axis is backwards.

In [ ]:
(t.pps_trials[t.pps_trials["usable"]]
   .groupby("position_rank")["rt_ms"]
   .agg(["count", "mean", "median"])
   .round(1))

# Aim 1 — The PPS effect in young healthy controls

## H1. Facilitation increases as the stimulus approaches

Supported if the bootstrap 95% CI on the near-far index lies entirely above zero.

In [ ]:
r_h1 = h1.run(t)

## H1a. The facilitation profile is sigmoidal

A sigmoid implies a *boundary*. A straight line does not. This compares flat,
linear and sigmoid fits by AIC.

In [ ]:
r_h1a = h1a.run(t)

## H2. The PPS boundary moves outward for faster approach

Note that H2 refuses to draw a sigmoid whose inflection falls outside the tested
range, or whose slope is implausible. If it says the fit was rejected, then
Delta_PPS is not measuring a boundary shift and H2 cannot be interpreted.

In [ ]:
r_h2 = h2.run(t)

## H3. The collision boundary shifts with ball speed

In [ ]:
r_h3 = h3.run(t)

## H4. Test-retest reliability

H4 produces the **SDC** for Delta_PPS, which is the noise floor H6b needs. In the
old notebook this passed silently through a global variable. Now it is an
explicit hand-off, so you cannot run H6b without H4 by accident.

In [ ]:
r_h4 = h4.run(t)

t.sdc_delta_pps = r_h4["SDC_DELTA_PPS"]
print(f"\nnoise floor carried forward to H6b: {t.sdc_delta_pps}")

# Aim 2 — PPS and its recalibration in Parkinson's disease

These need one patient and one matched control. Without them each cell prints
that it is skipped and returns `{"skipped": True}`, so the notebook still runs
top to bottom on pilot data.

## H5. Does the patient show distance-dependent facilitation at all?

In [ ]:
r_h5 = h5.run(t)

## H6. Is the patient's speed recalibration altered?

Two tests, and the second is **gated** on the first. If the patient shows no
recalibration signal at all (Test A), comparing their Delta_PPS to a control's
would just be comparing noise, so Test B is not run.

H6 also chooses which patient session to use. **H7 and H8b both depend on that
choice**, so it is handed over explicitly below.

In [ ]:
r_h6 = h6.run(t)

t.patient_session = r_h6.get("PATIENT_SESSION_USED")
t.control_session = r_h6.get("CONTROL_SESSION_USED", 1)
print(f"\nsessions carried forward: patient={t.patient_session}, control={t.control_session}")

## H6b. Does dopamine modulate the recalibration? (exploratory)

Requires the bootstrap to be consistent in sign **and** the effect to be bigger
than the H4 noise floor. A tight bootstrap on an unreliable measure is not
evidence.

In [ ]:
r_h6b = h6b.run(t)

## H7. Is the patient's collision-boundary shift altered?

In [ ]:
r_h7 = h7.run(t)

# Aim 3 — PPS plasticity and Hit-or-Miss judgment

## H8a. Across young controls

Exact permutation Spearman correlations. Be honest about power: at n = 5, a
one-sided exact p < 0.05 needs rho >= 0.9. A null result here means "we could not
have detected anything short of a near-perfect relationship", not "there is no
relationship".

In [ ]:
r_h8a = h8a.run(t)

## H8b. In the patient-control pair

Uses a **joint sign criterion**, not two separate tests. Two measures can each be
negative 96% of the time while being negative *together* only 60% of the time.
Since the claim is a co-occurrence, we measure the co-occurrence directly.

This needs H6's bootstrap, paired on the bootstrap index, which is why `r_h6` is
passed in rather than recomputed.

In [ ]:
r_h8b = h8b.run(t, r_h6)

## Summary

In [ ]:
summary = pd.DataFrame([
    {"hypothesis": "H1",  "outcome": r_h1["supported"]},
    {"hypothesis": "H1a", "outcome": f"{r_h1a['n_supported']}/{r_h1a['n_total']} participants"},
    {"hypothesis": "H2",  "outcome": r_h2["supported"]},
    {"hypothesis": "H3",  "outcome": r_h3["supported"]},
    {"hypothesis": "H5",  "outcome": r_h5.get("h5_supported_overall", "skipped")},
    {"hypothesis": "H6",  "outcome": r_h6.get("h6_classification", "skipped")},
    {"hypothesis": "H6b", "outcome": r_h6b.get("h6b_classification", "skipped")},
    {"hypothesis": "H7",  "outcome": r_h7.get("h7_classification", "skipped")},
    {"hypothesis": "H8a.1", "outcome": r_h8a["h8a_1"]["supported"]},
    {"hypothesis": "H8a.2", "outcome": r_h8a["h8a_2_near"]["supported"]},
    {"hypothesis": "H8a.3", "outcome": r_h8a["h8a_3"]["supported"]},
])
summary